# 9 — Diffusion: Synthetic SEP Generation

Trains a masked-attention DDPM on the 118 real SEP training sequences and
generates synthetic SEP samples, then assembles the training sets used by
notebook 10. This notebook **generates data only** — it trains no classifiers.

It is the diffusion counterpart of notebook 6 (TimeGAN) and follows the same
protocol and variant sizes, so the two generators are directly comparable.

**Input:** `final_split_data_HybridNorm_Tomek` (from notebook 3)

**Output:** `diffusion_datasets/{diffusion_balanced,diffusion_8000,diffusion_2000,diffusion_500}.pkl`
and `sep_samples/sep_real_vs_synthetic_diffusion.pkl`.

**Run after:** notebook 3. **Run before:** notebook 10.

## The generator

`./masked-ts-diffusion` — a compact DDPM whose denoiser is a Transformer with
causal self-attention, so position `t` can only attend to positions `<= t`.
Despite the causal mask it is an *unconditional full-sequence* model, not an
autoregressive forecaster: every reverse-diffusion step rewrites the whole
sequence.

## Runtime

Roughly 10 min training + 25 min sampling on Apple MPS at the default settings.
Set `FAST_RUN = True` for a quick pipeline check with throwaway sample quality.


# ⚠️ Environment Requirements

This notebook uses the diffusion code in `./masked-ts-diffusion` (PyTorch only).
Unlike notebook 6 it does **not** need TensorFlow, so it runs in the normal
project environment.

## Setup

Nothing to install for the generator itself. `./masked-ts-diffusion` holds two
flat modules, `api.py` and `model.py`, which the notebook puts on `sys.path`
and imports directly:

```python
sys.path.insert(0, os.path.abspath("./masked-ts-diffusion"))
from api import TimeSeriesDDPM
```

The folder name has hyphens, so it is not an importable package name — this is
why the modules are imported top-level rather than as `masked_ts_diffusion`.

## Notes
- `torch>=2.1` — the denoiser and diffusion loop
- `device="auto"` picks CUDA, then Apple MPS, then CPU
- No TensorFlow / Keras pinning, and no `timegan_env` kernel switch
- Sampling is chunked (`SAMPLE_BATCH`) because one `sample()` call for ~12k
  sequences of length 288 would allocate a 288x288 attention matrix per head
  for every sequence at once


## Diffusion Training and Dataset Generation

In [1]:
import pickle
import numpy as np

# ══════════════════════════════════════════════════════════════
# LOAD SAVED TOMEK-CLEANED HYBRID DATA
# Input folder: ./final_split_data_HybridNorm_Tomek
# Output variables:
#   X_train, y_train
#   X_val,   y_val
#   X_test,  y_test
# ══════════════════════════════════════════════════════════════

def load_split(path):
    with open(path, "rb") as f:
        data = pickle.load(f)
    return data["X"].astype(np.float32), data["y"]


TOMEK_DIR = "./final_split_data_HybridNorm_Tomek"

X_train, y_train = load_split(f"{TOMEK_DIR}/train_set.pkl")
X_val,   y_val   = load_split(f"{TOMEK_DIR}/val_set.pkl")
X_test,  y_test  = load_split(f"{TOMEK_DIR}/test_set.pkl")

print("Loaded Tomek-cleaned Hybrid dataset:")
print(f"Train: X={X_train.shape} | y={y_train.shape}")
print(f"Val  : X={X_val.shape} | y={y_val.shape}")
print(f"Test : X={X_test.shape} | y={y_test.shape}")

print("\nClass counts:")
print("Train:", np.bincount(y_train.astype(int), minlength=2))
print("Val  :", np.bincount(y_val.astype(int), minlength=2))
print("Test :", np.bincount(y_test.astype(int), minlength=2))

Loaded Tomek-cleaned Hybrid dataset:
Train: X=(12441, 288, 10) | y=(12441,)
Val  : X=(1780, 288, 10) | y=(1780,)
Test : X=(3559, 288, 10) | y=(3559,)

Class counts:
Train: [12323   118]
Val  : [1763   17]
Test : [3525   34]


In [5]:
import numpy as np
import pickle
import time
import os
import sys
import warnings

warnings.filterwarnings("ignore")

# ══════════════════════════════════════════════════════════════
# MASKED-ATTENTION DIFFUSION (DDPM) — FINAL DATASET GENERATION
# TOMEK-CLEANED HYBRID DATA
# Creates:
#   diffusion_balanced  -> no RUS, all Non-SEP, SEP matched to all Non-SEP
#   diffusion_8000
#   diffusion_2000
#   diffusion_500
#
# Same protocol as notebook 6 (TimeGAN); only the generator differs.
# ══════════════════════════════════════════════════════════════

# The diffusion code lives as flat modules in ./masked-ts-diffusion. That
# folder name contains hyphens, so it cannot be imported as a Python package --
# api.py and model.py are put on sys.path and imported as top-level modules.
DIFFUSION_PKG_DIR = os.path.abspath("./masked-ts-diffusion")
if DIFFUSION_PKG_DIR not in sys.path:
    sys.path.insert(0, DIFFUSION_PKG_DIR)

import torch
from api import TimeSeriesDDPM

print(f"torch {torch.__version__} | "
      f"mps={torch.backends.mps.is_available()} | "
      f"cuda={torch.cuda.is_available()}")


# ══════════════════════════════════════════════════════════════
# STEP 1 — PREPARE REAL SEP AND NON-SEP DATA
# ══════════════════════════════════════════════════════════════
# Assumes Tomek-cleaned data already loaded:
# X_train, y_train, X_val, y_val, X_test, y_test

y_int = y_train.astype(int)

idx_real_sep = np.where(y_int == 1)[0]
idx_nsep     = np.where(y_int == 0)[0]

n_sep_real = len(idx_real_sep)

X_real_sep = X_train[idx_real_sep].astype(np.float32)
X_nsep_all = X_train[idx_nsep].astype(np.float32)

print(f"\n  Real SEP samples     : {n_sep_real}")
print(f"  Real Non-SEP samples : {len(X_nsep_all)}")
print(f"  SEP shape            : {X_real_sep.shape}")

ori_data = X_real_sep.astype(np.float32)


# ══════════════════════════════════════════════════════════════
# STEP 2 — TRAIN THE DIFFUSION MODEL AND GENERATE SYNTHETIC SEP
# ══════════════════════════════════════════════════════════════

# FAST_RUN = True runs a quick end-to-end pipeline check (a couple of minutes)
# and produces poor samples. Set it to False for the real run.
FAST_RUN = False

if FAST_RUN:
    parameters = {
        "epochs": 20,
        "diffusion_steps": 50,
        "model_dim": 64,
        "num_layers": 2,
        "num_heads": 4,
        "batch_size": 32,
        "learning_rate": 2e-4,
        "seed": 42,
        "device": "auto",
        "verbose": True,
    }
else:
    parameters = {
        "epochs": 10000,
        "diffusion_steps": 1000,
        "model_dim": 144,
        "num_layers": 4,
        "num_heads": 4,
        "batch_size": 32,
        "learning_rate": 2e-4,
        "seed": 42,
        "device": "auto",
        "verbose": True,
    }

# Sampling is chunked. A single sample(12032) call would build a 288x288
# attention matrix per head for every sequence at once. Measured throughput on
# this data is flat between batch 64 and 128 and degrades above that.
SAMPLE_BATCH = 128

# Need enough synthetic SEP for:
# 1) diffusion_balanced: SEP must match all Non-SEP
# 2) diffusion_8000: SEP must reach 8000
n_generate_diffusion = max(
    8000 - n_sep_real,
    len(X_nsep_all) - n_sep_real
)

if n_generate_diffusion <= 0:
    raise ValueError(
        "No synthetic SEP samples are needed. "
        "Check SEP and Non-SEP counts."
    )

print(f"\n  Training the diffusion model on {n_sep_real} real SEP samples …")
print(f"  Config                           : {parameters}")
print(f"  Synthetic SEP samples to generate: {n_generate_diffusion}")

generator = TimeSeriesDDPM(parameters)

t0 = time.time()
generator.fit(ori_data)
total_train_time = time.time() - t0

print(f"\n  Training finished in {total_train_time:.2f}s "
      f"(final loss {generator.loss_history[-1]:.6f})")
print(f"  Sampling {n_generate_diffusion} sequences in chunks of {SAMPLE_BATCH} …")

t0 = time.time()
chunks = []
remaining = n_generate_diffusion

while remaining > 0:
    take = min(SAMPLE_BATCH, remaining)
    chunks.append(generator.sample(take))
    remaining -= take
    done = n_generate_diffusion - remaining
    print(f"    generated {done}/{n_generate_diffusion}", end="\r", flush=True)

infer_time = time.time() - t0

X_synthetic_all = np.concatenate(chunks, axis=0).astype(np.float32)
n_synthetic = len(X_synthetic_all)

del chunks

if not np.isfinite(X_synthetic_all).all():
    raise ValueError("Generated data contains NaN or Inf.")

if X_synthetic_all.shape[1:] != X_real_sep.shape[1:]:
    raise ValueError(
        f"Generated shape {X_synthetic_all.shape} does not match "
        f"real SEP shape {X_real_sep.shape}."
    )

print(f"\n  ✓ Total training time                    : {total_train_time:.2f}s")
print(f"  ✓ Inference time ({n_synthetic} samples) : {infer_time:.4f}s")
print(f"  ✓ Time per sample                        : {infer_time / n_synthetic * 1000:.4f}ms")
print(f"  ✓ Generated shape                        : {X_synthetic_all.shape}")

# Quick distribution sanity check against the real SEP data
print(f"\n  Per-feature mean/std (real vs synthetic):")
print(f"  {'feat':<6} {'real mean':>11} {'syn mean':>11} {'real std':>11} {'syn std':>11}")
feature_names = ['F', 'Np', 'P4', 'P5', 'P6', 'Tp', 'V', 'Vx', 'Xl', 'Xs']
for j, fname in enumerate(feature_names):
    print(f"  {fname:<6} "
          f"{X_real_sep[:, :, j].mean():>11.4f} "
          f"{X_synthetic_all[:, :, j].mean():>11.4f} "
          f"{X_real_sep[:, :, j].std():>11.4f} "
          f"{X_synthetic_all[:, :, j].std():>11.4f}")


# ══════════════════════════════════════════════════════════════
# STEP 3 — SAVE REAL VS SYNTHETIC SEP SAMPLES
# ══════════════════════════════════════════════════════════════

rng = np.random.default_rng(42)

n_compare = min(118, len(X_real_sep), len(X_synthetic_all))

idx_real_compare = rng.choice(
    len(X_real_sep),
    size=n_compare,
    replace=False
)

idx_synthetic_compare = rng.choice(
    len(X_synthetic_all),
    size=n_compare,
    replace=False
)

SAVE_DIR = "./sep_samples"
os.makedirs(SAVE_DIR, exist_ok=True)

save_path = os.path.join(SAVE_DIR, "sep_real_vs_synthetic_diffusion.pkl")

with open(save_path, "wb") as f:
    pickle.dump({
        "X_real_sep": X_real_sep[idx_real_compare],
        "y_real_sep": np.zeros(n_compare, dtype=np.int32),      # 0 = real

        "X_synthetic_sep": X_synthetic_all[idx_synthetic_compare],
        "y_synthetic_sep": np.ones(n_compare, dtype=np.int32),  # 1 = synthetic

        "total_train_time": total_train_time,
        "infer_time": infer_time,
        "n_sep_real": n_sep_real,
        "n_synthetic": n_synthetic,
        "n_compare": n_compare,
        "parameters": parameters,
        "loss_history": generator.loss_history,
    }, f)

print(f"\n✅ Saved real vs synthetic ({n_compare} each) → {os.path.abspath(save_path)}")
print(f"   real SEP      : {n_compare} samples  (label=0)")
print(f"   synthetic SEP : {n_compare} samples  (label=1)")


# ══════════════════════════════════════════════════════════════
# STEP 4 — BUILD DATASETS FROM SYNTHETIC SEP
# ══════════════════════════════════════════════════════════════

print(f"\n  Available Non-SEP samples : {len(X_nsep_all)}")
print(f"  Real SEP samples          : {n_sep_real}")
print(f"  Synthetic SEP samples     : {n_synthetic}")

sep_configs = [
    ("diffusion_balanced", len(X_nsep_all)),   # no RUS
    ("diffusion_8000",    8000),
    ("diffusion_2000",    2000),
    ("diffusion_500",      500),
]

diffusion_datasets = {}

print(
    f"\n{'Variant':<20} "
    f"{'NSEP':>8} "
    f"{'SEP':>8} "
    f"{'Real':>8} "
    f"{'Synthetic':>10} "
    f"{'RUS?':>8} "
    f"{'Shape'}"
)
print("─" * 90)

for cfg_name, n_sep in sep_configs:

    if n_sep > len(X_nsep_all):
        raise ValueError(
            f"{cfg_name}: requested {n_sep} Non-SEP samples, "
            f"but only {len(X_nsep_all)} are available."
        )

    if n_sep < n_sep_real:
        raise ValueError(
            f"{cfg_name}: target SEP count {n_sep} is smaller than "
            f"the number of real SEP samples {n_sep_real}."
        )

    n_synthetic_needed = n_sep - n_sep_real

    if n_synthetic_needed > len(X_synthetic_all):
        raise ValueError(
            f"{cfg_name}: requested {n_synthetic_needed} synthetic SEP samples, "
            f"but only {len(X_synthetic_all)} are available."
        )

    # -----------------------------------------------------
    # Select synthetic SEP
    # -----------------------------------------------------
    idx_syn = rng.choice(
        len(X_synthetic_all),
        size=n_synthetic_needed,
        replace=False
    )

    X_syn = X_synthetic_all[idx_syn]

    # Combine all real SEP + selected synthetic SEP
    X_sep = np.concatenate([X_real_sep, X_syn], axis=0).astype(np.float32)
    y_sep = np.ones(len(X_sep), dtype=np.float32)

    # -----------------------------------------------------
    # Select Non-SEP
    # diffusion_balanced uses all Non-SEP, no RUS
    # other variants randomly select n_sep Non-SEP
    # -----------------------------------------------------
    if cfg_name == "diffusion_balanced":
        X_ns = X_nsep_all.copy()
        use_rus = False
    else:
        idx_ns = rng.choice(
            len(X_nsep_all),
            size=n_sep,
            replace=False
        )
        X_ns = X_nsep_all[idx_ns]
        use_rus = True

    y_ns = np.zeros(len(X_ns), dtype=np.float32)

    # -----------------------------------------------------
    # Combine and shuffle
    # -----------------------------------------------------
    X_tr = np.concatenate([X_ns, X_sep], axis=0).astype(np.float32)
    y_tr = np.concatenate([y_ns, y_sep], axis=0)

    idx_shuf = rng.permutation(len(X_tr))
    X_tr = X_tr[idx_shuf]
    y_tr = y_tr[idx_shuf]

    print(
        f"{cfg_name:<20} "
        f"{len(X_ns):>8} "
        f"{len(X_sep):>8} "
        f"{n_sep_real:>8} "
        f"{n_synthetic_needed:>10} "
        f"{str(use_rus):>8} "
        f"{X_tr.shape}"
    )

    diffusion_datasets[cfg_name] = {
        "X_train": X_tr,
        "y_train": y_tr,

        "X_val": X_val,
        "y_val": y_val,

        "X_test": X_test,
        "y_test": y_test,

        "real_sep": n_sep_real,
        "synthetic_sep": n_synthetic_needed,
        "nsep": len(X_ns),
        "sep": len(X_sep),
        "use_rus": use_rus,
    }


# ══════════════════════════════════════════════════════════════
# STEP 5 — SAVE ALL DIFFUSION DATASETS
# ══════════════════════════════════════════════════════════════

DATASET_SAVE_DIR = "./diffusion_datasets"
os.makedirs(DATASET_SAVE_DIR, exist_ok=True)

for cfg_name, d in diffusion_datasets.items():

    save_path = os.path.join(DATASET_SAVE_DIR, f"{cfg_name}.pkl")

    with open(save_path, "wb") as f:
        pickle.dump(d, f)

    print(f"Saved {cfg_name} → {save_path}")

print(f"\n✅ All diffusion datasets saved to {os.path.abspath(DATASET_SAVE_DIR)}/")
print(f"   Total training time                  : {total_train_time:.2f}s")
print(f"   Inference time ({n_synthetic} samples): {infer_time:.4f}s")
print(f"   Time per sample                      : {infer_time / n_synthetic * 1000:.4f}ms")

print("\nDatasets created:")
for key in diffusion_datasets.keys():
    print(" ", key)


torch 2.12.0 | mps=True | cuda=False

  Real SEP samples     : 118
  Real Non-SEP samples : 12323
  SEP shape            : (118, 288, 10)

  Training the diffusion model on 118 real SEP samples …
  Config                           : {'epochs': 10000, 'diffusion_steps': 1000, 'model_dim': 144, 'num_layers': 4, 'num_heads': 4, 'batch_size': 32, 'learning_rate': 0.0002, 'seed': 42, 'device': 'auto', 'verbose': True}
  Synthetic SEP samples to generate: 12205
epoch 001/10000 loss=1.207419
epoch 100/10000 loss=0.118987
epoch 200/10000 loss=0.093255
epoch 300/10000 loss=0.099506
epoch 400/10000 loss=0.095688
epoch 500/10000 loss=0.089330
epoch 600/10000 loss=0.083110
epoch 700/10000 loss=0.075956
epoch 800/10000 loss=0.060875
epoch 900/10000 loss=0.065550
epoch 1000/10000 loss=0.075835
epoch 1100/10000 loss=0.058017
epoch 1200/10000 loss=0.068859
epoch 1300/10000 loss=0.081652
epoch 1400/10000 loss=0.094776
epoch 1500/10000 loss=0.071746
epoch 1600/10000 loss=0.068979
epoch 1700/10000 loss=0